# DentalGPT-7B-1026 — Kaggle deployment and experiment runner

This notebook is the **single runner/orchestrator notebook** for the project.

Runtime architecture:

```text
Panoramic X-ray
      ↓
llama.cpp multimodal server
      ├── DentalGPT-7B-1026 GGUF
      └── DentalGPT mmproj GGUF
      ↓
DentalGPTRunner.ask(image, question)
      ↓
BASIC / DISEASE_HIERARCHY / DISEASE_AND_LOCATION
      ↓
raw observations + deterministic atomic statuses
      ↓
optional text-only orchestrator
      ↓
saved JSON
```

Important design choices:

- Do **not** separately load `Qwen/Qwen2.5-VL-7B-Instruct`.
- Do **not** use Transformers or BitsAndBytes for the DentalGPT GGUF.
- Keep DentalGPT as the image-reasoning model.
- Keep JSON/report structuring outside DentalGPT.
- Start with `BASIC`, verify one real run, then move to deeper modes.


In [ ]:
# ============================================================
# CELL 1 — Python dependencies
# ============================================================
# llama.cpp itself is compiled later with CUDA.
# We intentionally do not install transformers / bitsandbytes / qwen-vl-utils.

%pip install -q \
    "huggingface_hub>=0.26" \
    "openai>=1.55" \
    "pydantic>=2.7" \
    "requests>=2.31" \
    "pillow>=10.0"

print("Python dependencies installed.")


In [ ]:
# ============================================================
# CELL 2 — Locate/import the project files
# ============================================================
# Supported Kaggle layouts:
# A) /kaggle/working/dentalgpt_project_rewrite
# B) dentalgpt_project_rewrite.zip added as a Kaggle dataset
# C) current directory contains the required Python files

import os
import sys
import json
import time
import shutil
import zipfile
import subprocess
from pathlib import Path

REQUIRED_PROJECT_FILES = {
    "dentalgpt.py",
    "llama_runtime.py",
    "pipeline.py",
    "prompts.py",
}

WORK_PROJECT_DIR = Path("/kaggle/working/dentalgpt_project_rewrite")


def has_project_files(path: Path) -> bool:
    return path.is_dir() and REQUIRED_PROJECT_FILES.issubset(
        {p.name for p in path.iterdir() if p.is_file()}
    )


project_candidates = [WORK_PROJECT_DIR, Path.cwd()]
PROJECT_DIR = next((p for p in project_candidates if has_project_files(p)), None)

# If modules are not directly available, try the project ZIP from /kaggle/input.
if PROJECT_DIR is None and Path("/kaggle/input").exists():
    zip_hits = list(Path("/kaggle/input").rglob("dentalgpt_project_rewrite.zip"))
    if zip_hits:
        project_zip = zip_hits[0]
        print("Found project ZIP:", project_zip)
        with zipfile.ZipFile(project_zip, "r") as zf:
            zf.extractall("/kaggle/working")
        if has_project_files(WORK_PROJECT_DIR):
            PROJECT_DIR = WORK_PROJECT_DIR

# Last fallback: search /kaggle/input for the modules themselves.
if PROJECT_DIR is None and Path("/kaggle/input").exists():
    for dentalgpt_file in Path("/kaggle/input").rglob("dentalgpt.py"):
        candidate = dentalgpt_file.parent
        if has_project_files(candidate):
            PROJECT_DIR = candidate
            break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project modules. Add dentalgpt_project_rewrite.zip "
        "to the Kaggle notebook as a dataset, or place dentalgpt.py, "
        "llama_runtime.py, pipeline.py and prompts.py under "
        "/kaggle/working/dentalgpt_project_rewrite."
    )

PROJECT_DIR = PROJECT_DIR.resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("PROJECT_DIR =", PROJECT_DIR)

from dentalgpt import DentalGPTRunner
from llama_runtime import LlamaCppServer, build_llama_cpp, download_dentalgpt, find_llama_server
from pipeline import DentalAnalysisPipeline, LLMOrchestrator
from prompts import broad_records

print("Project imports succeeded.")


In [ ]:
# ============================================================
# CELL 3 — MAIN EXPERIMENT CONFIGURATION
# ============================================================
# This is the main cell to edit between experiments.

# Input / output
IMAGE_PATH = "/kaggle/input/YOUR_DATASET/YOUR_IMAGE.jpg"
OUTPUT_DIR = "/kaggle/working/dental_outputs"

# BASIC: 4 broad screening calls.
# DISEASE_HIERARCHY: broad + family + 14 atomic calls.
# DISEASE_AND_LOCATION: hierarchy + location follow-ups.
ANALYSIS_MODE = "BASIC"

RUN_SMOKE_TEST = True
RUN_PIPELINE = True
SHOW_IMAGE = True

# DentalGPT repository
HF_REPO_ID = "mradermacher/DentalGPT-7B-1026-GGUF"
MODEL_DIR = "/kaggle/working/models/dentalgpt"

# QUALITY = Q6_K + F16 mmproj, preferred baseline on a 16 GB P100.
# FAST    = Q4_K_M + Q8 mmproj, faster/lower-memory development preset.
MODEL_PRESET = "QUALITY"  # "QUALITY" | "FAST"

if MODEL_PRESET.upper() == "QUALITY":
    MODEL_FILENAME = "DentalGPT-7B-1026.Q6_K.gguf"
    MMPROJ_FILENAME = "DentalGPT-7B-1026.mmproj-f16.gguf"
elif MODEL_PRESET.upper() == "FAST":
    MODEL_FILENAME = "DentalGPT-7B-1026.Q4_K_M.gguf"
    MMPROJ_FILENAME = "DentalGPT-7B-1026.mmproj-Q8_0.gguf"
else:
    raise ValueError("MODEL_PRESET must be QUALITY or FAST")

# llama.cpp runtime
LLAMA_CPP_DIR = "/kaggle/working/llama.cpp"
LLAMA_CPP_REF = "b10516"
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8080
SERVER_ALIAS = "dentalgpt"
SERVER_LOG_PATH = "/kaggle/working/llama_dentalgpt_server.log"
N_GPU_LAYERS = 999
CTX_SIZE = 8192
PARALLEL = 1
CUDA_ARCH = None  # None = auto-detect; P100 fallback is 60
BUILD_JOBS = 4
SERVER_STARTUP_TIMEOUT = 300.0

# DentalGPT generation defaults
DEFAULT_MAX_TOKENS = 768
TEMPERATURE = 0.0
TOP_P = 1.0
SEED = 0
REQUEST_TIMEOUT_SECONDS = 600.0
CACHE_PROMPT = False
LOCATE_UNCERTAIN = True

# Optional external text-only orchestrator
USE_OPENAI_ORCHESTRATOR = False
ORCHESTRATOR_MODEL = None
ORCHESTRATOR_BASE_URL = None
ORCHESTRATOR_TIMEOUT_SECONDS = 600.0
ORCHESTRATOR_MAX_RETRIES = 2



print("Configuration:")
print("  mode         =", ANALYSIS_MODE)
print("  preset       =", MODEL_PRESET)
print("  model        =", MODEL_FILENAME)
print("  mmproj       =", MMPROJ_FILENAME)
print("  context      =", CTX_SIZE)
print("  orchestrator =", USE_OPENAI_ORCHESTRATOR)


In [ ]:
# ============================================================
# CELL 4 — Environment diagnostics
# ============================================================

import platform

print("Python:", platform.python_version())
print("Platform:", platform.platform())

for executable in ["git", "cmake", "nvcc", "nvidia-smi"]:
    print(f"{executable:12s}:", shutil.which(executable))

print("\nGPU:")
subprocess.run(["nvidia-smi"], check=False)


def detect_cuda_arch(fallback: str = "60") -> str:
    try:
        output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            text=True,
            stderr=subprocess.STDOUT,
        )
        first = output.strip().splitlines()[0].strip()
        arch = first.replace(".", "")
        if arch.isdigit():
            return arch
    except Exception as exc:
        print("CUDA architecture auto-detection failed:", exc)

    print(f"Falling back to CUDA architecture {fallback}.")
    return fallback


CUDA_ARCH_RESOLVED = str(CUDA_ARCH) if CUDA_ARCH else detect_cuda_arch("60")
print("\nCUDA_ARCH_RESOLVED =", CUDA_ARCH_RESOLVED)

if not shutil.which("cmake"):
    raise RuntimeError("cmake is required to build llama.cpp.")
if not shutil.which("git"):
    raise RuntimeError("git is required to obtain llama.cpp.")
if not shutil.which("nvcc"):
    raise RuntimeError("nvcc was not found. Enable a GPU accelerator in Kaggle.")


In [ ]:
# ============================================================
# CELL 5 — Kaggle secrets
# ============================================================
# HF_TOKEN is normally optional because the GGUF repo is public.
# OPENAI_API_KEY is needed only for the optional orchestrator.

HF_TOKEN = HF_TOKEN
ORCHESTRATOR_API_KEY = ORCHESTRATOR_API_KEY

if USE_OPENAI_ORCHESTRATOR:
    if not ORCHESTRATOR_MODEL:
        raise ValueError("Set ORCHESTRATOR_MODEL when USE_OPENAI_ORCHESTRATOR=True.")
    if not ORCHESTRATOR_API_KEY:
        raise ValueError("OPENAI_API_KEY is required when USE_OPENAI_ORCHESTRATOR=True.")

print("HF token configured:", bool(HF_TOKEN))
print("External orchestrator enabled:", USE_OPENAI_ORCHESTRATOR)


In [ ]:
# ============================================================
# CELL 6 — Build/find pinned llama.cpp with CUDA
# ============================================================

LLAMA_SERVER = find_llama_server()

if LLAMA_SERVER is None:
    print("llama-server not found; building pinned llama.cpp...")
    LLAMA_SERVER = build_llama_cpp(
        source_dir=LLAMA_CPP_DIR,
        cuda_arch=CUDA_ARCH_RESOLVED,
        jobs=BUILD_JOBS,
        ref=LLAMA_CPP_REF,
    )
else:
    print("Found existing llama-server:", LLAMA_SERVER)

LLAMA_SERVER = Path(LLAMA_SERVER).resolve()
if not LLAMA_SERVER.is_file():
    raise FileNotFoundError(LLAMA_SERVER)

print("llama-server =", LLAMA_SERVER)


In [ ]:
# ============================================================
# CELL 7 — Download the exact DentalGPT GGUF + mmproj
# ============================================================

model_files = download_dentalgpt(
    model_dir=MODEL_DIR,
    repo_id=HF_REPO_ID,
    model_filename=MODEL_FILENAME,
    mmproj_filename=MMPROJ_FILENAME,
    hf_token=HF_TOKEN,
)

MODEL_PATH = Path(model_files.model_path).resolve()
MMPROJ_PATH = Path(model_files.mmproj_path).resolve()

print("Language model:")
print(" ", MODEL_PATH)
print(f"  size = {MODEL_PATH.stat().st_size / (1024**3):.2f} GiB")

print("\nVision projector:")
print(" ", MMPROJ_PATH)
print(f"  size = {MMPROJ_PATH.stat().st_size / (1024**3):.2f} GiB")


In [ ]:
# ============================================================
# CELL 8 — Start a clean DentalGPT llama.cpp server
# ============================================================
# Rerunning this cell first stops the server object owned by this notebook.
# Then /v1/models is checked so we do not accidentally use another model.

import requests

if "server" in globals():
    try:
        server.stop()
    except Exception as exc:
        print("Previous server cleanup:", exc)

server = LlamaCppServer(
    binary=LLAMA_SERVER,
    model_path=MODEL_PATH,
    mmproj_path=MMPROJ_PATH,
    host=SERVER_HOST,
    port=SERVER_PORT,
    alias=SERVER_ALIAS,
    n_gpu_layers=N_GPU_LAYERS,
    ctx_size=CTX_SIZE,
    parallel=PARALLEL,
    startup_timeout=SERVER_STARTUP_TIMEOUT,
    log_path=SERVER_LOG_PATH,
)

server.start(reuse_existing=False)

models_response = requests.get(f"{server.base_url}/v1/models", timeout=10)
models_response.raise_for_status()
models_payload = models_response.json()
model_ids = [
    item.get("id")
    for item in models_payload.get("data", [])
    if isinstance(item, dict)
]

print("Server URL:", server.base_url)
print("Server model IDs:", model_ids)

if SERVER_ALIAS not in model_ids:
    raise RuntimeError(
        f"Expected alias {SERVER_ALIAS!r}, but /v1/models returned {model_ids}. "
        f"Inspect {SERVER_LOG_PATH}."
    )

print("DentalGPT llama.cpp server verified.")


In [ ]:
# ============================================================
# CELL 9 — Construct the DentalGPT runner
# ============================================================

dental_runner = DentalGPTRunner(
    base_url=server.base_url,
    api_model=SERVER_ALIAS,
    model_id=f"{HF_REPO_ID}:{MODEL_FILENAME}",
    max_tokens=DEFAULT_MAX_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    seed=SEED,
    timeout=REQUEST_TIMEOUT_SECONDS,
    cache_prompt=CACHE_PROMPT,
)

print("DentalGPTRunner ready.")
print("model_id =", dental_runner.model_id)


In [ ]:
# ============================================================
# CELL 10 — Validate and preview the input radiograph
# ============================================================

from PIL import Image
from IPython.display import display

image_path = Path(IMAGE_PATH)
if not image_path.is_file():
    raise FileNotFoundError(
        f"IMAGE_PATH does not exist:\n{image_path}\n\n"
        "Edit IMAGE_PATH in CELL 3 before continuing."
    )

image = Image.open(image_path)
print("Image:", image_path)
print("Format:", image.format)
print("Mode:", image.mode)
print("Size:", image.size)

if SHOW_IMAGE:
    display(image)


In [ ]:
# ============================================================
# CELL 11 — One real production-prompt smoke test
# ============================================================
# Verifies image encoding, mmproj, multimodal formatting and generation.
# Uses the first real BASIC prompt rather than a separate demo prompt.

smoke = None

if RUN_SMOKE_TEST:
    smoke_record = broad_records()[0]

    print("question_id:", smoke_record["question_id"])
    print("layer:", smoke_record["layer"])
    print("\nQUESTION\n--------")
    print(smoke_record["question"])

    smoke = dental_runner.ask(
        IMAGE_PATH,
        smoke_record["question"],
        max_tokens=smoke_record.get("max_tokens"),
    )

    print("\nRAW DENTALGPT RESPONSE\n----------------------")
    print(smoke["raw_answer"])

    print("\nMETADATA")
    print("  latency_seconds    =", smoke.get("latency_seconds"))
    print("  finish_reason      =", smoke.get("finish_reason"))
    print("  truncated          =", smoke.get("truncated"))
    print("  prompt_tokens      =", smoke.get("prompt_tokens"))
    print("  completion_tokens  =", smoke.get("completion_tokens"))

    if smoke.get("truncated"):
        print(
            "\nWARNING: Smoke test hit the token limit. "
            "Increase that prompt's max_tokens before interpreting its answer."
        )
else:
    print("RUN_SMOKE_TEST=False; skipped.")


In [ ]:
# ============================================================
# CELL 12 — Build the analysis pipeline
# ============================================================

orchestrator = None
if USE_OPENAI_ORCHESTRATOR:
    orchestrator = LLMOrchestrator(
        model=ORCHESTRATOR_MODEL,
        base_url=ORCHESTRATOR_BASE_URL,
        api_key=ORCHESTRATOR_API_KEY,
        timeout=ORCHESTRATOR_TIMEOUT_SECONDS,
        max_retries=ORCHESTRATOR_MAX_RETRIES,
    )

pipeline = DentalAnalysisPipeline(
    dental_runner=dental_runner,
    orchestrator=orchestrator,
    locate_uncertain=LOCATE_UNCERTAIN,
)

print("Pipeline ready.")
print("Analysis mode:", ANALYSIS_MODE)
print("Locate UNCERTAIN findings:", LOCATE_UNCERTAIN)
print("External orchestrator:", bool(orchestrator))


In [ ]:
# ============================================================
# CELL 13 — Orchestrator connectivity smoke test
# ============================================================
# This sends text only and does not run the DentalGPT model or pipeline.

if orchestrator is None:
    print("External orchestrator is disabled; skipped.")
else:
    smoke_completion = orchestrator.client.chat.completions.create(
        model=orchestrator.model,
        messages=[{"role": "user", "content": "Reply with exactly: ORCHESTRATION_OK"}],
    )
    print("Orchestrator smoke response:", smoke_completion.choices[0].message.content)


In [ ]:
# ============================================================
# CELL 14 — Run the selected analysis mode
# ============================================================

result = None

if RUN_PIPELINE:
    result = pipeline.run(
        image_path=IMAGE_PATH,
        mode=ANALYSIS_MODE,
        output_dir=OUTPUT_DIR,
    )

    print("\nRUN COMPLETE")
    print("------------")
    print("Saved to:", result["saved_to"])
    print("DentalGPT model:", result["dentalgpt_model"])
    print("DentalGPT calls:", result["dentalgpt_call_count"])
    print("Total latency:", result["total_latency_seconds"], "seconds")
else:
    print("RUN_PIPELINE=False; skipped.")


In [ ]:
# ============================================================
# CELL 15 — Compact result summary
# ============================================================

import pandas as pd
from IPython.display import display

if result is None:
    print("No pipeline result. Run CELL 13 first.")
else:
    atomic_rows = []
    for item in result["observations"]:
        if item.get("layer") == "ATOMIC_FINDING":
            atomic_rows.append(
                {
                    "condition": item.get("target"),
                    "status": item.get("parsed_status"),
                    "latency_s": item.get("latency_seconds"),
                    "finish_reason": item.get("finish_reason"),
                    "truncated": item.get("truncated"),
                }
            )

    if atomic_rows:
        print("Atomic findings:")
        display(pd.DataFrame(atomic_rows))
    else:
        print("No atomic findings in this run. That is expected in BASIC mode.")

    location_rows = []
    for item in result["observations"]:
        if item.get("layer") == "LOCATION":
            location_rows.append(
                {
                    "condition": item.get("target"),
                    "question_id": item.get("question_id"),
                    "answer": item.get("parsed_answer"),
                    "latency_s": item.get("latency_seconds"),
                    "truncated": item.get("truncated"),
                }
            )

    if location_rows:
        print("\nLocation follow-ups:")
        display(pd.DataFrame(location_rows))

    if result.get("orchestrated_output"):
        print("\nFinal orchestrated report:")
        print(result["orchestrated_output"]["report"])


In [ ]:
# ============================================================
# CELL 16 — Inspect raw observations / prompt debugging
# ============================================================

if result is None:
    print("No pipeline result. Run CELL 13 first.")
else:
    for i, item in enumerate(result["observations"], start=1):
        print("\n" + "=" * 100)
        print(
            f"{i}/{len(result['observations'])} | "
            f"{item.get('question_id')} | "
            f"{item.get('layer')} | "
            f"target={item.get('target')}"
        )
        print("-" * 100)
        print("QUESTION:")
        print(item.get("question"))
        print("\nRAW ANSWER:")
        print(item.get("raw_answer"))
        print("\nPARSED:")
        print(
            item.get("parsed_status")
            if item.get("layer") == "ATOMIC_FINDING"
            else item.get("parsed_answer")
        )
        print(
            "latency=", item.get("latency_seconds"),
            "| finish=", item.get("finish_reason"),
            "| truncated=", item.get("truncated"),
        )


In [ ]:
# ============================================================
# CELL 17 — Server diagnostics
# ============================================================
# Run this if model loading or inference fails.

log_path = Path(SERVER_LOG_PATH)
if log_path.is_file():
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-120:]))
else:
    print("No server log found at:", log_path)


In [ ]:
# ============================================================
# CELL 18 — Optional cleanup
# ============================================================
# Stop only when completely finished. Keep the server running during
# experiments so the model remains loaded in GPU memory.

# server.stop()
# print("DentalGPT server stopped.")
